In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
import shutil
import numpy as np
from tqdm import tqdm
from glob import glob
from src.service.fragment.net import Net
from skl2onnx.helpers.onnx_helper import load_onnx_model
from skl2onnx.helpers.onnx_helper import save_onnx_model
from src.service.fragment.create_fragments import get_fragments
from src.utilities.change_model_layers_dimension import change_layers_dimension

In [3]:
x = np.random.randn(1, 3, 224, 224).astype(np.float32)

nets = []
model_names = sorted(glob('../_models/*'))
for i, model in tqdm(enumerate(model_names), position=0, leave=True):
    print(model)
    model_onnx1 = load_onnx_model(model)
    change_layers_dimension(model_onnx1)
    fragments1 = get_fragments(model_onnx1, x)
    net1 = Net(fragments1, i)
    nets.append(net1)

0it [00:00, ?it/s]

../_models/alexnet.onnx


1it [00:06,  6.71s/it]

../_models/densenet121.onnx


2it [00:26, 14.52s/it]

...

3it [00:28,  8.70s/it]

../_models/resnet50.onnx


4it [00:43, 11.09s/it]

../_models/vgg16.onnx


5it [01:11, 14.38s/it]


In [4]:
for net in nets:
    print("number of fragments", len(net))

number of fragments 8
number of fragments 5
number of fragments 13
number of fragments 6
number of fragments 16


In [5]:
shutil.rmtree("../_results_without_finetune/fragments", ignore_errors=True)
os.makedirs("../_results_without_finetune/fragments", exist_ok=True)
for i,net in enumerate(nets):
    for j,fragment in enumerate(net):
        folder = f"../_results_without_finetune/fragments/net{i:03}/"
        os.makedirs(folder, exist_ok = True)
        filename = folder+f'fragment{j:03}.onnx'
        save_onnx_model(fragment.fragment, filename)